# <font color="#418FDE" size="6.5" uppercase>**Generalized Bases**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Match generalized linear models to target distributions such as counts and positive skewed values. 
- Interpret link functions, deviance, and evaluation choices for generalized models. 
- Design polynomial, spline, and interaction bases to capture nonlinear patterns with linear estimators. 


## **1. Generalized Regression Models**

### **1.1. Poisson Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_01_01.jpg?v=1783842197" width="250">



>* Models counts within fixed observation units.
>* Better than linear regression for event frequencies.

>* Predictors change counts multiplicatively, keeping predictions positive.
>* Exposure terms enable fair event-rate comparisons.

>* Check assumptions: overdispersion and excess zeros.
>* Poisson remains a useful starting model.



In [ ]:
#@title Python Code - Poisson Regression

# Poisson regression models nonnegative event counts.
# This example compares counts with linear trends.
# We visualize expected counts and observed events.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create a small exposure feature.
hours = np.linspace(1, 8, 24)

# Define a log link relationship.
rate = np.exp(0.25 + 0.22 * hours)

# Sample count outcomes safely.
counts = rng.poisson(rate)

# Check matching array lengths.
assert hours.shape == counts.shape

# Fit a straight line baseline.
linear_fit = np.polyfit(hours, counts, 1)

# Fit log mean for Poisson idea.
log_fit = np.polyfit(hours, np.log(counts + 0.5), 1)

# Convert back to positive counts.
poisson_curve = np.exp(np.polyval(log_fit, hours))

# Build linear predictions for contrast.
linear_curve = np.polyval(linear_fit, hours)

# Print a few teaching takeaways.
print("Observed counts are whole numbers.")
print("Poisson-style predictions stay positive.")
print("Linear regression can dip unrealistically low.")
print(f"Smallest linear prediction: {linear_curve.min():.2f}")
print(f"Smallest Poisson-style prediction: {poisson_curve.min():.2f}")

# Plot counts and both fitted curves.
plt.figure(figsize=(7, 4))
plt.scatter(hours, counts, color="black", label="Observed counts")
plt.plot(hours, linear_curve, label="Straight-line fit", linewidth=2)
plt.plot(hours, poisson_curve, label="Poisson-style fit", linewidth=2)

# Label the chart clearly.
plt.xlabel("Store open hours")
plt.ylabel("Customer arrivals")
plt.title("Poisson Regression Keeps Expected Counts Positive")
plt.legend()

# Show the final teaching figure.
plt.tight_layout(); plt.show()



### **1.2. Choosing Target Distributions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_01_02.jpg?v=1783842203" width="250">



>* Match distributions to outcome shape and limits.
>* Wrong choices mislead predictions and interpretation.

>* Use count models for event frequencies.
>* Check variance and excess zeros carefully.

>* Use positive skew distributions for right-tailed outcomes.
>* Check fit, bounds, and realistic assumptions.



In [ ]:
#@title Python Code - Choosing Target Distributions

# This script compares outcome distribution choices.
# It uses tiny simulated teaching data.
# The example stays simple and visual.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Simulate count outcomes from Poisson.
counts = rng.poisson(lam=3.0, size=300)

# Simulate positive skewed continuous outcomes.
costs = rng.gamma(shape=2.0, scale=120.0, size=300)

# Summarize the count outcome briefly.
count_mean = counts.mean()
count_var = counts.var()

# Summarize the positive skewed outcome.
cost_mean = costs.mean()
cost_min = costs.min()

# Check basic support constraints safely.
assert counts.min() >= 0, "Counts must stay nonnegative."
assert cost_min > 0, "Positive outcomes must stay above zero."

# Print short interpretation lines.
print("Counts example: clinic arrivals per hour")
print(f"Mean={count_mean:.2f}, Variance={count_var:.2f}")
print("Good starting family: Poisson for counts")
print("Positive example: repair costs in dollars")
print(f"Mean={cost_mean:.2f}, Minimum={cost_min:.2f}")
print("Good starting family: Gamma for positive skew")
print("Normal regression may predict impossible negatives")

# Create one figure with two panels.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Plot the count distribution shape.
axes[0].hist(counts, bins=np.arange(-0.5, 11.5, 1), color="steelblue")
axes[0].set_title("Count outcome")
axes[0].set_xlabel("Events")
axes[0].set_ylabel("Frequency")

# Plot the positive skewed distribution shape.
axes[1].hist(costs, bins=18, color="darkorange")
axes[1].set_title("Positive skewed outcome")
axes[1].set_xlabel("Cost")
axes[1].set_ylabel("Frequency")

# Finish the layout and show.
plt.tight_layout()
plt.show()



### **1.3. Positive Skew Targets**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_01_03.jpg?v=1783842205" width="250">



>* Positive outcomes are often strongly right skewed.
>* Generalized models respect positivity and data shape.

>* Gamma models fit positive, mean-variance increasing outcomes.
>* Log links ensure positivity and proportional effects.

>* Check positivity, skewness, and possible zeros.
>* Choose models by fit, predictions, interpretability.



In [ ]:
#@title Python Code - Positive Skew Targets

# Positive targets often need special regression choices.
# This example compares linear and log scale fits.
# We use tiny synthetic right skewed data.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic seed.
rng = np.random.default_rng(7)
# Create one predictor column.
x = np.linspace(1.0, 10.0, 80)

# Generate positive skewed outcomes.
noise = rng.gamma(shape=2.0, scale=0.35, size=x.size)
# Mean grows multiplicatively with distance.
y = np.exp(1.0 + 0.22 * x) * noise

# Validate matching one dimensional shapes.
assert x.shape == y.shape
# Fit an ordinary linear trend.
linear_coef = np.polyfit(x, y, 1)

# Fit a log mean trend.
log_coef = np.polyfit(x, np.log(y), 1)
# Predict on the original scale.
linear_pred = np.polyval(linear_coef, x)

# Back transform log predictions.
gamma_like_pred = np.exp(np.polyval(log_coef, x))
# Check prediction sizes safely.
assert linear_pred.size == y.size

# Summarize positivity and skew.
print(f"Minimum target value: {y.min():.2f}")
print(f"Maximum target value: {y.max():.2f}")
print(f"Linear fit minimum prediction: {linear_pred.min():.2f}")
print(f"Log scale fit minimum prediction: {gamma_like_pred.min():.2f}")

# Explain the modeling lesson.
print("Positive skew suggests a positive mean model.")
print("A log link keeps fitted means above zero.")

# Draw one compact comparison plot.
plt.figure(figsize=(7, 4))
plt.scatter(x, y, s=22, alpha=0.65, label="Observed cost")
plt.plot(x, linear_pred, linewidth=2, label="Linear regression style")
plt.plot(x, gamma_like_pred, linewidth=2, label="Log link style")

# Label the axes clearly.
plt.xlabel("Route length in miles")
plt.ylabel("Delivery cost in dollars")
plt.title("Positive skewed targets favor positive fitted means")
plt.legend()

# Show the teaching figure.
plt.tight_layout()
plt.show()



## **2. Generalized Model Evaluation**

### **2.1. Understanding Link Functions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_02_01.jpg?v=1783842207" width="250">



>* Links connect linear predictors to constrained outcomes.
>* They keep predictions valid, like probabilities.

>* Coefficients act first on transformed scale.
>* Response changes depend on baseline and context.

>* Links encode assumptions and boundary behavior.
>* Choose links for realism and interpretation.



In [ ]:
#@title Python Code - Understanding Link Functions

# This script illustrates common generalized link functions.
# It compares linear and response scales simply.
# One plot shows how mappings differ.

# Import small numerical plotting tools.
import numpy as np
import matplotlib.pyplot as plt

# Build a simple linear predictor grid.
eta = np.linspace(-4.0, 4.0, 200)
prob = 1.0 / (1.0 + np.exp(-eta))
count = np.exp(eta)

# Choose a few coefficient changes.
etas = np.array([-2.0, 0.0, 2.0])
shift = 0.5
new_probs = 1.0 / (1.0 + np.exp(-(etas + shift)))

# Compute probability changes after shifting.
prob_changes = new_probs - (1.0 / (1.0 + np.exp(-etas)))
count_ratio = np.exp(shift)

# Print short interpretation messages.
print("Linear shift added to eta:", shift)
print("Logit probabilities before:", np.round(1.0 / (1.0 + np.exp(-etas)), 3))
print("Logit probability changes:", np.round(prob_changes, 3))
print("Log link count multiplier:", round(count_ratio, 3))
print("Same eta shift, different probability effects.")
print("Same eta shift, same count ratio.")

# Plot response curves for two links.
plt.figure(figsize=(7, 4))
plt.plot(eta, prob, label="Logit inverse: probability")
plt.plot(eta, count / count.max(), label="Log inverse scaled")
plt.axvline(0.0, color="gray", linestyle="--")

# Add labels and a compact title.
plt.xlabel("Linear predictor eta")
plt.ylabel("Response scale")
plt.title("Link functions map eta differently")
plt.legend()

# Show the single teaching plot.
plt.tight_layout()
plt.show()



### **2.2. Understanding Deviance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_02_02.jpg?v=1783842211" width="250">



>* Deviance measures model fit by distribution.
>* Small deviance fits well; large misses patterns.

>* Deviance respects each outcome's distribution.
>* Residual versus null deviance shows predictor value.

>* Low deviance helps, but is not enough.
>* Also check generalization, fairness, and calibration.



In [ ]:
#@title Python Code - Understanding Deviance

# Deviance compares fitted and ideal model likelihoods.
# This example uses tiny count data.
# We compare null and fitted deviance.

import numpy as np
import pandas as pd

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create tiny count data.
days = np.arange(1, 13)
promo = np.array([0, 0, 1, 1] * 3)

# Build counts with a pattern.
base = 8 + 0.6 * days + 4 * promo
counts = rng.poisson(base)

data = pd.DataFrame({
    "day": days, "promo": promo, "count": counts
})

# Check the tiny dataset size.
assert data.shape == (12, 3)

# Define Poisson deviance safely.
def poisson_deviance(y, mu):
    mu = np.clip(np.asarray(mu), 1e-9, None)
    y = np.asarray(y)

    # Handle zero counts safely.
    term = np.where(y == 0, 0.0, y * np.log(y / mu))
    return 2 * np.sum(term - (y - mu))

# Build a null prediction.
null_mu = np.full(len(data), data["count"].mean())

# Build a simple fitted prediction.
fitted_mu = 7.5 + 0.7 * data["day"] + 3.5 * data["promo"]
fitted_mu = np.clip(fitted_mu.to_numpy(), 1e-9, None)

# Compute both deviances.
null_dev = poisson_deviance(data["count"], null_mu)
resid_dev = poisson_deviance(data["count"], fitted_mu)

# Summarize the comparison.
improvement = null_dev - resid_dev
ratio = resid_dev / null_dev

# Print a short teaching summary.
print("Observed counts:", data["count"].tolist())
print("Null deviance:", round(null_dev, 2))
print("Residual deviance:", round(resid_dev, 2))
print("Deviance drop:", round(improvement, 2))
print("Residual/null ratio:", round(ratio, 2))
print("Smaller deviance means closer fit.")
print("Big drops suggest predictors explain structure.")



### **2.3. Evaluation Choices**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_02_03.jpg?v=1783842209" width="250">



>* Choose metrics based on model purpose.
>* Check fit, prediction, calibration, and complexity.

>* Prefer held-out validation for prediction performance.
>* Check ranking and probability calibration too.

>* Balance fit, simplicity, and practical usefulness.
>* Check subgroup fairness, not just overall scores.



In [ ]:
#@title Python Code - Evaluation Choices

# Evaluation depends on goals and outcome types.
# This example compares several generalized model checks.
# We use tiny synthetic binary outcome data.

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create one feature with nonlinear risk.
x = np.linspace(-3, 3, 120)
true_logit = -0.4 + 1.2 * x

# Convert logits into probabilities.
true_prob = 1 / (1 + np.exp(-true_logit))
y = rng.binomial(1, true_prob)

# Split into train and test parts.
train_x, test_x = x[:80], x[80:]
train_y, test_y = y[:80], y[80:]

# Reshape for scikit style estimators.
train_x = train_x.reshape(-1, 1)
test_x = test_x.reshape(-1, 1)

# Check that split sizes are valid.
assert train_x.shape[0] > 10 and test_x.shape[0] > 10
assert train_x.shape[1] == 1 and test_x.shape[1] == 1

# Fit a simple logistic regression model.
model = LogisticRegression(random_state=0, max_iter=200)
model.fit(train_x, train_y)

# Predict probabilities on both datasets.
train_p = model.predict_proba(train_x)[:, 1]
test_p = model.predict_proba(test_x)[:, 1]

# Compute common generalized model evaluations.
train_dev = 2 * log_loss(train_y, train_p, normalize=False)
test_dev = 2 * log_loss(test_y, test_p, normalize=False)
auc = roc_auc_score(test_y, test_p)

# Compute a simple calibration style score.
brier = brier_score_loss(test_y, test_p)
mean_pred = float(np.mean(test_p))
mean_obs = float(np.mean(test_y))

# Print a compact teaching summary.
print("Generalized model evaluation choices")
print(f"Train deviance: {train_dev:.2f}")
print(f"Test deviance:  {test_dev:.2f}")
print(f"Test ROC AUC:   {auc:.2f}")
print(f"Test Brier:     {brier:.3f}")
print(f"Mean predicted: {mean_pred:.3f}")
print(f"Mean observed:  {mean_obs:.3f}")
print("Use deviance for fit, AUC for ranking,")
print("and calibration checks for probability quality.")



## **3. Basis Design**

### **3.1. Polynomial Bases**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_03_01.jpg?v=1783842213" width="250">



>* Polynomial features let linear models fit curves.
>* Flexibility comes from transformed inputs, not estimator.

>* Choose degree for smoothness, stability, interpretability.
>* Judge overall curve, not individual coefficients.

>* Scale inputs and compare polynomial degrees carefully.
>* Bases encode smooth, realistic nonlinear assumptions.



In [ ]:
#@title Python Code - Polynomial Bases

# Polynomial bases create curved fits from linear models.
# This example compares straight and quadratic basis designs.
# A single plot shows why extra basis terms help.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic seed.
rng = np.random.default_rng(7)

# Create one predictor and curved target.
x = np.linspace(-3.0, 3.0, 60)
y = 1.5 + 0.8 * x + 1.2 * x**2

y = y + rng.normal(0.0, 1.2, size=x.shape)

# Validate matching shapes first.
assert x.shape == y.shape, "Shapes must match."

# Build a linear basis matrix.
X1 = np.column_stack([np.ones_like(x), x])

# Build a quadratic basis matrix.
X2 = np.column_stack([np.ones_like(x), x, x**2])

# Fit both models with least squares.
beta1 = np.linalg.lstsq(X1, y, rcond=None)[0]

beta2 = np.linalg.lstsq(X2, y, rcond=None)[0]

# Predict using both basis designs.
yhat1 = X1 @ beta1

yhat2 = X2 @ beta2

# Compare average squared errors.
mse1 = np.mean((y - yhat1) ** 2)

mse2 = np.mean((y - yhat2) ** 2)

# Print a short interpretation.
print(f"Linear basis MSE: {mse1:.2f}")
print(f"Quadratic basis MSE: {mse2:.2f}")
print("Quadratic basis adds x squared for curvature.")
print("The estimator stays linear in coefficients.")

# Draw data and both fitted curves.
plt.figure(figsize=(8, 5))
plt.scatter(x, y, s=28, alpha=0.7, label="Observed data")

plt.plot(x, yhat1, linewidth=2, label="Linear basis")
plt.plot(x, yhat2, linewidth=2, label="Quadratic basis")

# Label the plot clearly.
plt.title("Polynomial Bases Capture Curved Patterns")
plt.xlabel("Study hours, centered scale")
plt.ylabel("Exam score trend")

plt.legend()
plt.tight_layout()
plt.show()



### **3.2. Spline Bases**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_03_02.jpg?v=1783842215" width="250">



>* Splines model curves with smooth local pieces.
>* They capture changing effects without unstable polynomials.

>* Splines create local features, keeping linear estimation.
>* Knots control flexibility and smooth transitions.

>* Local splines model nonlinear patterns smoothly.
>* Choose knots for edges, smoothness, clarity.



In [ ]:
#@title Python Code - Spline Bases

# Spline bases model smooth local nonlinear patterns.
# This example builds spline features manually.
# A linear fit then follows curved data.

import numpy as np
import matplotlib.pyplot as plt

# Use a deterministic seed here.
rng = np.random.default_rng(7)
# Create one predictor with curvature.
x = np.linspace(0.0, 10.0, 120)
# Build a smooth target pattern.
y_true = 2.0 + 0.5 * x + np.sin(x)

# Add small noise for realism.
y = y_true + rng.normal(0.0, 0.18, x.size)
# Choose interior knot locations.
knots = np.array([3.0, 6.0, 8.0])
# Confirm matching sample sizes.
assert x.shape[0] == y.shape[0]

# Build a truncated power spline basis.
X = np.column_stack([
    np.ones_like(x),
    x,
    x ** 2,
    x ** 3,
    np.maximum(0.0, x - knots[0]) ** 3,
    np.maximum(0.0, x - knots[1]) ** 3,
    np.maximum(0.0, x - knots[2]) ** 3,
])

# Confirm design matrix dimensions.
assert X.shape == (x.size, 7)
# Solve least squares coefficients.
coef, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
# Compute fitted spline values.
y_spline = X @ coef

# Fit one straight line for comparison.
line_coef = np.polyfit(x, y, 1)
# Evaluate the straight line.
y_line = np.polyval(line_coef, x)
# Summarize basis size briefly.
print(f"Basis columns: {X.shape[1]}")
# Report a simple error comparison.
print(f"Line RMSE: {np.sqrt(np.mean((y - y_line) ** 2)):.3f}")

# Report spline fit improvement.
print(f"Spline RMSE: {np.sqrt(np.mean((y - y_spline) ** 2)):.3f}")
# Explain what knots do.
print(f"Knots at x = {knots.tolist()}")
# Plot data and both fits.
plt.figure(figsize=(8, 4.8))

# Show noisy observations lightly.
plt.scatter(x, y, s=18, alpha=0.55, label="Observed data")
# Show the true hidden curve.
plt.plot(x, y_true, linewidth=2, label="True pattern")
# Show the straight linear fit.
plt.plot(x, y_line, linewidth=2, label="Straight line")
# Show the spline-based fit.
plt.plot(x, y_spline, linewidth=2.5, label="Spline basis fit")

# Mark knot positions visually.
for knot in knots:
    plt.axvline(knot, color="gray", linestyle="--", alpha=0.45)

# Add a clear teaching title.
plt.title("Spline bases let linear models bend smoothly")
# Label the horizontal axis.
plt.xlabel("Predictor x")
# Label the vertical axis.
plt.ylabel("Target y")
# Add a compact legend.
plt.legend()

# Improve spacing for readability.
plt.tight_layout()
# Display the single required plot.
plt.show()



### **3.3. Interaction Bases**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_03_03.jpg?v=1783842217" width="250">



>* Interactions let one feature affect another.
>* Combined features add flexibility, coefficients stay linear.

>* Interactions model effects that vary jointly.
>* More flexibility increases complexity and interpretation challenges.

>* Balance interactions for fit, clarity, and stability.
>* Combine plausible interactions with splines thoughtfully.



In [ ]:
#@title Python Code - Interaction Bases

# Interaction terms let one feature modify another.
# This script builds a simple interaction basis.
# A contour plot shows changing fitted slopes.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic seed.
rng = np.random.default_rng(7)

# Create two continuous predictors.
study_hours = rng.uniform(0.0, 10.0, 80)
preparation = rng.uniform(0.0, 5.0, 80)

# Build an outcome with interaction.
noise = rng.normal(0.0, 1.2, 80)
score = 40 + 3 * study_hours + 2 * preparation
score = score + 1.5 * study_hours * preparation + noise

# Check matching sample sizes.
assert study_hours.shape == preparation.shape == score.shape

# Build additive design matrix.
X_add = np.column_stack([
    np.ones_like(study_hours),
    study_hours,
    preparation,
])

# Build interaction design matrix.
X_int = np.column_stack([
    np.ones_like(study_hours),
    study_hours,
    preparation,
    study_hours * preparation,
])

# Fit both linear models.
coef_add, _, _, _ = np.linalg.lstsq(X_add, score, rcond=None)
coef_int, _, _, _ = np.linalg.lstsq(X_int, score, rcond=None)

# Compute training mean squared errors.
pred_add = X_add @ coef_add
pred_int = X_int @ coef_int
mse_add = np.mean((score - pred_add) ** 2)
mse_int = np.mean((score - pred_int) ** 2)

# Print a short interpretation.
print(f"Additive MSE: {mse_add:.2f}")
print(f"Interaction MSE: {mse_int:.2f}")
print(f"Interaction weight: {coef_int[3]:.2f}")
print("Lower MSE shows the interaction basis helps.")

# Create a prediction grid.
hours_grid = np.linspace(0.0, 10.0, 60)
prep_grid = np.linspace(0.0, 5.0, 60)
H, P = np.meshgrid(hours_grid, prep_grid)

# Predict with the interaction model.
Z = (
    coef_int[0]
    + coef_int[1] * H
    + coef_int[2] * P
    + coef_int[3] * H * P
)

# Plot the fitted response surface.
plt.figure(figsize=(7, 5))
contours = plt.contourf(H, P, Z, levels=18, cmap="viridis")
plt.scatter(study_hours, preparation, c=score, edgecolor="black", s=35)

# Label the basis plot.
plt.xlabel("Study hours")
plt.ylabel("Preparation level")
plt.title("Interaction basis creates a curved response surface")
plt.colorbar(contours, label="Predicted score")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Generalized Bases**</font>


In this lecture, you learned to:
- Match generalized linear models to target distributions such as counts and positive skewed values. 
- Interpret link functions, deviance, and evaluation choices for generalized models. 
- Design polynomial, spline, and interaction bases to capture nonlinear patterns with linear estimators. 

In the next Module (Module 11), we will go over 'None'